# 0. Imports & Config Read

In [92]:
# External Dependencies
import pandas as pd
from pprint import pprint

# Internal Imports
from src.data_collectors.collector_yfin import YFinanceNSEPipeline
from src.modelling.var_engine import LeadLagVAREngine
from src.signal_generators.var_masked_signal import GrangerCausalityMaskedSignal
from src.utils.file_operators import load_yaml

In [93]:
# Load the catalog and params from yaml to dict
config_catalog = load_yaml("../config_catalog.yml")
config_params = load_yaml("../config_parameters.yml")

## 0.1. Parameters

In [94]:
pprint(config_params)

{'adf_testing_params': {'alpha': 0.05},
 'granger_causal_signal_gen_params': {'max_position_size': 1.0,
                                      'min_log_returns_threshold': 0.0005,
                                      'target_volatility': 0.15,
                                      'volatility_scaling': True},
 'market_params': {'end_time': '15:30',
                   'start_time': '09:30',
                   'time_zone': 'Asia/Kolkata'},
 'time_params': {'interval': '1m', 'period': '7d'},
 'universe_params': {'tech_eqs': ['TCS', 'INFY', 'MPHASIS', 'LTIM', 'COFORGE'],
                     'universe_name': 'tech_eqs'},
 'var_params': {'alpha': 0.05,
                'estimation_method': 'ols',
                'ic_criterion': 'aic',
                'max_lags': 10}}


## 0.2. Catalog

In [95]:
pprint(config_catalog)

{'data': {'collector_yfin_cleaned_data': {'directory': 'C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/collector_yfin/01_processed',
                                          'file_format': 'csv',
                                          'file_name': 'cleaned_data',
                                          'versioned': True},
          'collector_yfin_log_returns': {'directory': 'C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/collector_yfin/01_processed',
                                         'file_format': 'csv',
                                         'file_name': 'log_returns_data',
                                         'versioned': True},
          'collector_yfin_raw_data': {'directory': 'C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/collector_yfin/00_raw_data',
                                      'file_format': 'csv',
                                      'file_name': 'raw_

# 1. Data Load and QC

In [96]:
# Initialise a data loader obj
data_loader = YFinanceNSEPipeline(
    config_catalog=config_catalog,
    config_params=config_params
)

In [97]:
# Fetch Data
data_loader.fetch_raw_data()

INFO: [INGESTION]: Fetching 7d of 1m data for 5 assets from Yahoo Finance (Sequential Mode)
INFO: [INGESTION]: Downloading TCS (TCS.NS)...
SUCCESS:   └─ Received 2526 bars for TCS
INFO: [INGESTION]: Downloading INFY (INFY.NS)...
SUCCESS:   └─ Received 2526 bars for INFY
INFO: [INGESTION]: Downloading MPHASIS (MPHASIS.NS)...
SUCCESS:   └─ Received 2478 bars for MPHASIS
INFO: [INGESTION]: Downloading LTIM (LTIM.NS)...


$LTIM.NS: No data found, symbol may be delisted


WARN:   └─ Warning: Empty or invalid payload for LTIM
INFO: [INGESTION]: Downloading COFORGE (COFORGE.NS)...
SUCCESS:   └─ Received 2525 bars for COFORGE
SUCCESS: [INGESTION]: Successfully downloaded 2527 raw rows across 4 valid assets from Yahoo Finance


,TCS,INFY,MPHASIS,COFORGE
Datetime,,,,
2026-08-18 09:15:00+05:30,2303.000000,1127.000000,2492.699951,1787.900024
2026-08-18 09:16:00+05:30,2294.000000,1122.599976,2487.199951,1781.300049
2026-08-18 09:17:00+05:30,2295.399902,1122.500000,2491.899902,1785.400024
2026-08-18 09:18:00+05:30,2294.899902,1121.500000,2490.000000,1784.699951
2026-08-18 09:19:00+05:30,2292.699951,1121.000000,2490.000000,1785.599976
...,...,...,...,...
2026-08-26 15:11:00+05:30,2270.100098,1122.699951,2398.699951,1883.400024
2026-08-26 15:12:00+05:30,2271.399902,1122.199951,2399.000000,1884.300049
2026-08-26 15:13:00+05:30,2272.100098,1122.800049,2403.500000,1884.099976


In [98]:
# Align and Process Data
data_loader.align_and_clean_data()

INFO: [PROCESSING]: Processing 7d of 1m data for 5 assets
SUCCESS: [PROCESSING] Aligned grid size: 2422 rows across 5 stocks.


,TCS,INFY,MPHASIS,COFORGE
Datetime,,,,
2026-08-18 09:30:00+05:30,2293.500000,1119.699951,2479.600098,1781.800049
2026-08-18 09:31:00+05:30,2292.300049,1119.699951,2477.899902,1782.400024
2026-08-18 09:32:00+05:30,2292.500000,1120.000000,2478.300049,1779.699951
2026-08-18 09:33:00+05:30,2293.100098,1119.000000,2478.100098,1779.699951
2026-08-18 09:34:00+05:30,2293.000000,1119.099976,2481.100098,1783.900024
...,...,...,...,...
2026-08-26 15:11:00+05:30,2270.100098,1122.699951,2398.699951,1883.400024
2026-08-26 15:12:00+05:30,2271.399902,1122.199951,2399.000000,1884.300049
2026-08-26 15:13:00+05:30,2272.100098,1122.800049,2403.500000,1884.099976


In [99]:
# Compute Log Returns
data_loader.compute_log_returns()

INFO: [PROCESSING]: Computing Log Returns for 5 assets


,TCS,INFY,MPHASIS,COFORGE
Datetime,,,,
2026-08-18 09:31:00+05:30,-0.000523,0.000000,-0.000686,0.000337
2026-08-18 09:32:00+05:30,0.000087,0.000268,0.000161,-0.001516
2026-08-18 09:33:00+05:30,0.000262,-0.000893,-0.000081,0.000000
2026-08-18 09:34:00+05:30,-0.000044,0.000089,0.001210,0.002357
2026-08-18 09:35:00+05:30,0.000262,0.000357,-0.000121,-0.000561
...,...,...,...,...
2026-08-26 15:11:00+05:30,0.000000,0.000267,-0.000417,0.000797
2026-08-26 15:12:00+05:30,0.000572,-0.000445,0.000125,0.000478
2026-08-26 15:13:00+05:30,0.000308,0.000535,0.001874,-0.000106


In [100]:
# Perform ADF test
data_loader.validate_stationarity()

INFO: [ANALYSING]: Performing ADF for 5 assets


,ADF Statistic,p-value,Stationary (I(0))
TCS,-52.1376,0.0,True
INFY,-54.4471,0.0,True
MPHASIS,-27.3526,0.0,True
COFORGE,-17.1338,0.0,True


# 2. Fit and Model VAR

In [101]:
# Initialise a data loader obj
var_engine = LeadLagVAREngine(
    config_catalog=config_catalog,
    config_params=config_params
)

In [102]:
# Fit the VAR
var_engine.fit_var_model()

SUCCESS: [VAR ENGINE]: Selected optimal lag p = 3 minute(s) via 'AIC' criterion.
INFO: [VAR Engine]: Fitted VAR(3) using OLS.


{'optimal_lag': np.int64(3),
 'method': 'ols',
 'phi_matrice_path': WindowsPath('C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/var_engine/01_weights/phi_matrices_tech_eqs_20260826_225455.csv'),
 'phi_matrie':                  TCS          INFY   MPHASIS   COFORGE
 const      -0.000002  2.899308e-07 -0.000014  0.000032
 L1.TCS     -0.136543  1.060236e-01  0.023765 -0.071921
 L1.INFY     0.026146 -1.870122e-01  0.012116  0.124057
 L1.MPHASIS  0.042901  3.642597e-02 -0.118334 -0.005867
 L1.COFORGE  0.050099  8.594723e-02  0.073856 -0.046043
 L2.TCS     -0.050508  4.877540e-03 -0.039445 -0.144572
 L2.INFY     0.007933 -7.421703e-03  0.056143  0.027456
 L2.MPHASIS  0.090278  3.600341e-02  0.035014  0.282715
 L2.COFORGE -0.058702 -3.763677e-02 -0.062295 -0.118960
 L3.TCS      0.016605  4.657366e-02  0.010948 -0.010553
 L3.INFY    -0.002150 -1.030526e-03  0.016593  0.051688
 L3.MPHASIS  0.026007 -2.015318e-02  0.025979  0.082599
 L3.COFORGE -0.015096  1.661334

In [103]:
# Compute to checck grangers causality
var_engine.compute_grangers_causality()

INFO: [VAR ENGINE]: Computing pairwise Granger Causality tests across 4 assets...
SUCCESS: [VAR ENGINE]: Successfully computed Granger Causality p-value matrix.

       GRANGER CAUSALITY QUANTITATIVE INTERPRETATION (Alpha = 0.05)

[1] PAIRWISE DIRECTIONAL BREAKDOWN:

[VALID LINK]    TCS vs INFY:
    - p(TCS -> INFY): 0.024493
    - p(INFY -> TCS): 0.57376
    - Classification : STRICT LEAD-LAG
    - Action         : Trade INFY using TCS forecast

[VALID LINK]    TCS vs MPHASIS:
    - p(TCS -> MPHASIS): 0.530174
    - p(MPHASIS -> TCS): 4e-06
    - Classification : STRICT LEAD-LAG (REVERSE)
    - Action         : Trade TCS using MPHASIS forecast

[FEEDBACK LOOP] TCS vs COFORGE:
    - p(TCS -> COFORGE): 0.002424
    - p(COFORGE -> TCS): 3e-06
    - Classification : FEEDBACK LOOP
    - Action         : BLOCK / REDUCE WEIGHT (Systemic Co-movement)

[NO LINK]       INFY vs MPHASIS:
    - p(INFY -> MPHASIS): 0.160868
    - p(MPHASIS -> INFY): 0.202408
    - Classification : INDEPENDENT
    -

,TCS,INFY,MPHASIS,COFORGE
TCS,1.000000,0.024493,0.530174,0.002424
INFY,0.573760,1.000000,0.160868,0.000026
MPHASIS,0.000004,0.202408,1.000000,0.000000
COFORGE,0.000003,0.000111,0.000011,1.000000


In [104]:
# Forecast using base VAR model
var_engine.generate_next_bar_signal()

SUCCESS: [VAR ENGINE]: Successfully generated 1-step-ahead return forecasts for 'tech_eqs'.


,TCS,INFY,MPHASIS,COFORGE
forecast_return_t_plus_1,-0.000245,0.000051,-0.000496,-0.00078


# 3. Use masked causality and generate signal

In [105]:
# Pass master config_params and config_catalog
signal_generator = GrangerCausalityMaskedSignal(config_params=config_params, config_catalog=config_catalog)

In [106]:
# Zero-argument call: Automatically resolves all CSV files from catalog paths & persists output
target_weights_df, audit_metadata = signal_generator.generate_positions()

INFO: [SignalGenerator]: Loading forecasts from 'forecast_signals'...
{'directory': 'C:/Users/sharv/Documents/Sharvil/Projects/lead-lag-vector-auto-regression/data/var_engine/04_forecasts', 'file_name': 'forecast_signals', 'file_format': 'csv', 'versioned': True}
INFO: [SignalGenerator]: Loading Granger matrix from 'grangers_causality'...
INFO: [SignalGenerator]: Loading historical log returns from 'log_returns_data'...
SUCCESS: [SignalGenerator]: Exported target weights to: C:\Users\sharv\Documents\Sharvil\Projects\lead-lag-vector-auto-regression\data\var_engine\05_target_signals\target_position_weights_tech_eqs_20260826_225455.csv
SUCCESS: [SignalGenerator]: Exported target metadata to: C:\Users\sharv\Documents\Sharvil\Projects\lead-lag-vector-auto-regression\data\var_engine\05_target_signals\target_position_metadata_tech_eqs_20260826_225455.csv
